# 🏆 VISTA — AI Semantic Segmentation Pipeline
## Tahap 2: Physical Environment Assessment (Computer Vision)

**Tim FiveHonk! — WebGIS Competition MAPID 2026**

---

Notebook ini menganalisis citra Google Street View menggunakan model **SegFormer** untuk mengekstraksi indikator visual:

| Indikator | Deskripsi | Kelas Cityscapes |
|---|---|---|
| **Sky View Factor (SVF)** | Proporsi langit terlihat | Class 10: Sky |
| **Green View Index (GVI)** | Proporsi vegetasi/pohon | Class 8: Vegetation |
| **Road Width Index** | Proporsi jalan | Class 0: Road |
| **Street Canyon Enclosure** | Proporsi bangunan | Class 2: Building |
| **Sidewalk Ratio** | Proporsi trotoar | Class 1: Sidewalk |

**Referensi Proposal:** Tabel 4, View-based Streetscape Perception (Physical)

## 1. Setup Environment & Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# ⚠️ SESUAIKAN PATH INI DENGAN LOKASI FOLDER ANDA DI DRIVE
# ============================================================
BASE_PATH = '/content/drive/MyDrive/ai_pipeline'
# ============================================================

import os
IMAGE_DIR = os.path.join(BASE_PATH, 'data', 'images')
OUTPUT_DIR = os.path.join(BASE_PATH, 'data')

if os.path.exists(IMAGE_DIR):
    images = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    # Filter gambar yang terlalu kecil (kemungkinan dummy/placeholder)
    real_images = [f for f in images if os.path.getsize(os.path.join(IMAGE_DIR, f)) > 5000]
    print(f'📁 Folder ditemukan: {IMAGE_DIR}')
    print(f'🖼️  Total file gambar: {len(images)}')
    print(f'✅ Gambar valid (>5KB): {len(real_images)}')
    if len(real_images) == 0:
        print('\n⚠️  PERINGATAN: Semua gambar berukuran < 5KB (kemungkinan dummy/placeholder).')
        print('   Pastikan Anda sudah menjalankan scraper dengan API Key yang valid:')
        print('   python 2_scrape_gsv.py --api-key YOUR_KEY --max-images 100')
else:
    print(f'❌ Folder tidak ditemukan: {IMAGE_DIR}')
    print('   Pastikan Anda sudah meng-upload folder ai_pipeline ke Google Drive.')

## 2. Install Dependencies

In [ ]:
!pip install -q transformers torch torchvision Pillow matplotlib pandas tqdm

## 3. Load Model SegFormer (Cityscapes)

Model: `nvidia/segformer-b2-finetuned-cityscapes-1024-1024`

Cityscapes dataset mengenali 19 kelas elemen perkotaan. Kita hanya mengambil 5 kelas yang relevan untuk VISTA.

In [ ]:
import torch
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from tqdm import tqdm

# Gunakan model yang lebih akurat (b2) daripada b0
MODEL_NAME = 'nvidia/segformer-b2-finetuned-cityscapes-1024-1024'

print(f'Loading model: {MODEL_NAME}...')
processor = SegformerImageProcessor.from_pretrained(MODEL_NAME)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_NAME)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()
print(f'✅ Model loaded on {device}')

# Cityscapes class mapping untuk VISTA
VISTA_CLASSES = {
    'road_width_index':       0,   # Road
    'sidewalk_ratio':         1,   # Sidewalk
    'street_canyon_enclosure': 2,  # Building
    'green_view_index':       8,   # Vegetation
    'sky_view_factor':       10,   # Sky
}

# Warna untuk visualisasi
VISTA_COLORS = {
    0: [128, 64, 128],   # Road - ungu
    1: [244, 35, 232],   # Sidewalk - pink
    2: [70, 70, 70],     # Building - abu
    8: [107, 142, 35],   # Vegetation - hijau
    10: [70, 130, 180],  # Sky - biru
}

## 4. Proses Semantic Segmentation

In [ ]:
def segment_image(image_path):
    """Proses satu gambar dan kembalikan proporsi kelas VISTA."""
    try:
        image = Image.open(image_path).convert('RGB')
    except Exception as e:
        return None

    inputs = processor(images=image, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = torch.nn.functional.interpolate(
        outputs.logits,
        size=image.size[::-1],
        mode='bilinear',
        align_corners=False,
    )
    
    seg_map = logits.argmax(dim=1)[0].cpu().numpy()
    total_pixels = seg_map.size
    
    metrics = {}
    for metric_name, class_id in VISTA_CLASSES.items():
        ratio = np.sum(seg_map == class_id) / total_pixels
        metrics[metric_name] = round(ratio, 4)
    
    return metrics, seg_map, image

# Proses semua gambar
results = []
use_images = real_images if len(real_images) > 0 else images[:10]

print(f'Memproses {len(use_images)} gambar...')
for img_file in tqdm(use_images, desc='Segmentasi'):
    img_path = os.path.join(IMAGE_DIR, img_file)
    result = segment_image(img_path)
    if result:
        metrics, _, _ = result
        metrics['filename'] = img_file
        results.append(metrics)

df_results = pd.DataFrame(results)

# Hitung Visual Perception Score (Composite)
# Sesuai proposal: integrasi Sky View, Green View, Road Width, Street Canyon
df_results['visual_perception_score'] = (
    df_results['green_view_index'] * 0.30 +
    df_results['sky_view_factor'] * 0.25 +
    df_results['sidewalk_ratio'] * 0.20 +
    (1 - df_results['street_canyon_enclosure']) * 0.15 +
    df_results['road_width_index'] * 0.10
).round(4)

print(f'\n✅ Selesai! {len(results)} gambar berhasil diproses.')
print(f'\nRata-rata indikator:')
for col in VISTA_CLASSES.keys():
    print(f'  {col}: {df_results[col].mean():.4f}')
print(f'  visual_perception_score: {df_results["visual_perception_score"].mean():.4f}')

## 5. Visualisasi Hasil (Contoh 4 Gambar)

In [ ]:
# Tampilkan contoh segmentasi untuk 4 gambar
n_show = min(4, len(use_images))
fig, axes = plt.subplots(n_show, 2, figsize=(14, 4 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

for i in range(n_show):
    img_path = os.path.join(IMAGE_DIR, use_images[i])
    result = segment_image(img_path)
    if result:
        metrics, seg_map, original = result
        
        # Gambar asli
        axes[i, 0].imshow(original)
        axes[i, 0].set_title(f'Original: {use_images[i][:40]}', fontsize=10)
        axes[i, 0].axis('off')
        
        # Segmentation map dengan warna VISTA
        colored = np.zeros((*seg_map.shape, 3), dtype=np.uint8)
        for class_id, color in VISTA_COLORS.items():
            colored[seg_map == class_id] = color
        
        axes[i, 1].imshow(colored)
        gvi = metrics['green_view_index']
        svf = metrics['sky_view_factor']
        axes[i, 1].set_title(f'Segmentation | GVI={gvi:.2f} SVF={svf:.2f}', fontsize=10)
        axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'segmentation_preview.png'), dpi=150)
plt.show()
print('Preview disimpan ke data/segmentation_preview.png')

## 6. Simpan Hasil ke CSV

In [ ]:
output_csv = os.path.join(OUTPUT_DIR, 'physical_environment_score.csv')
df_results.to_csv(output_csv, index=False)

print(f'📊 Hasil analisis Visual Perception disimpan ke:')
print(f'   {output_csv}')
print(f'\n📋 Preview data:')
df_results.head(10)

## 7. Distribusi Skor
Histogram distribusi Visual Perception Score untuk seluruh TAS-Nits yang sudah diproses.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('VISTA - Distribusi Indikator Physical Environment', fontsize=14, fontweight='bold')

indicators = list(VISTA_CLASSES.keys()) + ['visual_perception_score']
colors = ['#8B5CF6', '#EC4899', '#6B7280', '#22C55E', '#3B82F6', '#F59E0B']

for idx, (col, color) in enumerate(zip(indicators, colors)):
    ax = axes[idx // 3, idx % 3]
    ax.hist(df_results[col], bins=20, color=color, alpha=0.7, edgecolor='white')
    ax.set_title(col.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('Score')
    mean_val = df_results[col].mean()
    ax.axvline(mean_val, color='red', linestyle='--', alpha=0.8)
    ax.text(mean_val, ax.get_ylim()[1]*0.9, f'μ={mean_val:.3f}', color='red', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'physical_env_distribution.png'), dpi=150)
plt.show()
print('Grafik distribusi disimpan.')

---
## ✅ Selesai!

File output yang dihasilkan:
- `data/physical_environment_score.csv` — Skor Physical Environment per gambar
- `data/segmentation_preview.png` — Contoh visual segmentasi
- `data/physical_env_distribution.png` — Distribusi skor

File `physical_environment_score.csv` nantinya akan digabung dengan `accessibility_score.csv` untuk membentuk **Urban Vitality Index (UVI)** pada Tahap 5.